In [92]:
import pandas as pd
import wrds
import networkx as nx
import matplotlib.pyplot as plt
import os
import requests

In [72]:
db = wrds.Connection(wrds_username='dheerajtls')

Loading library list...
Done


In [ ]:
p = r"D:\dissertation\data\TNIC\tnic_all_data\tnicall2023.txt"

df = pd.read_csv(p, sep=None, engine='python')
print(df.columns.tolist())
print(df.head())

['score', 'gvkey1', 'gvkey2', 'ball', 'year']
    score  gvkey1  gvkey2  ball  year
0     NaN    1004    1004   NaN  2023
1  0.0397    1004    1045   1.0  2023
2  0.0589    1004    1050   1.0  2023
3  0.0003    1004    1075   1.0  2023
4  0.0006    1004    1076   1.0  2023


In [51]:
comp = db.raw_sql("""
    SELECT gvkey, datadate, conm, cik, naicsh, sich, at, sale
    FROM comp.funda
    WHERE datadate BETWEEN '2023-01-01' AND '2023-12-31'
      AND indfmt='INDL' AND datafmt='STD'
      AND popsrc='D' AND consol='C'
""")
comp = comp.drop_duplicates(subset=['cik','gvkey'])
comp.to_csv("../data/compustat_2023.csv", index=False)

In [52]:
comp.dtypes

gvkey        string
datadate     string
conm         string
cik          string
naicsh        Int64
sich          Int64
at          Float64
sale        Float64
dtype: object

In [53]:
firms = set(df.gvkey1.unique()).union(set(df.gvkey2.unique()))
comp['gvkey'] = comp.gvkey.astype(int)

m = comp[comp.gvkey.isin(firms)].copy()
print(len(m), "matched of", len(firms))
print(m.cik.isna().sum(), "missing CIK")
print(m.naicsh.isna().sum(), "missing NAICS")
print(m.gvkey.duplicated().sum(), "duplicate gvkeys")

4099 matched of 4099
11 missing CIK
27 missing NAICS
0 duplicate gvkeys


In [58]:
m[m['cik'].isna()]

,gvkey,datadate,conm,cik,naicsh,sich,at,sale
264,4409,2023-12-31,ENZON PHARMACEUTICALS -OLD,<NA>,325414,2836,47.702,0.0
334,5179,2023-12-31,GLATFELTER CORP,<NA>,322120,2621,1563.796,1385.516
9724,137351,2023-12-31,BERKSHIRE HILLS BANCORP INC,<NA>,522110,6020,12430.821,619.081
10758,174024,2023-12-31,UNITED STATES OIL FUND LP,<NA>,523910,6799,1586.713,-34.9
11550,181911,2023-12-31,PROSHARES ULTRA GOLD,<NA>,<NA>,<NA>,<NA>,<NA>
11645,183303,2023-12-31,ABRDN PHYSICAL SILVER SH ETF,<NA>,<NA>,<NA>,<NA>,<NA>
11718,183869,2023-12-31,UNITED STATES 12 NAT GAS LP,<NA>,<NA>,<NA>,<NA>,<NA>
12001,185775,2023-12-31,ABRDN PHYSICAL PR M B SH ETF,<NA>,<NA>,<NA>,<NA>,<NA>
12178,187164,2023-12-31,HOMESTREET INC,<NA>,522110,6020,9392.45,441.664
12284,189511,2023-12-31,TEUCRIUM SOYBEAN FUND,<NA>,<NA>,<NA>,<NA>,<NA>


In [55]:
m2_df = comp[~comp.gvkey.isin(firms)]

In [56]:
m2_df['cik'].value_counts()

cik
0001230869    1
0000001961    1
0000002062    1
0000002230    1
0000351483    1
             ..
0002011053    1
0002032966    1
0002035989    1
0002071668    1
0002081206    1
Name: count, Length: 3867, dtype: Int64

In [57]:
comp[comp.cik.isin(['0001744780'])]

,gvkey,datadate,conm,cik,naicsh,sich,at,sale
4248,34159,2023-03-31,RIV CAPITAL INC,0001744780,523999,6799,261.818,6.807


In [37]:
comp.shape

(12708, 8)

In [60]:
df.score.describe()

count    1.663428e+07
mean     2.045238e-02
std      4.577612e-02
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      2.120000e-02
max      9.224000e-01
Name: score, dtype: float64

In [61]:
# seeds from the previous step
seed_keys = set(df.gvkey1.astype(int))

# TNIC-3 equivalent: threshold the pairwise scores
THRESH = 0.15          # placeholder — calibration will set this
e = df[(df.score >= THRESH)]

# one step out from seeds
nbrs = set(e[e.gvkey1.isin(seed_keys)].gvkey2)
universe = seed_keys | nbrs
print(len(seed_keys), "seeds ->", len(universe), "universe")

# subgraph on the universe
sub = e[e.gvkey1.isin(universe) & e.gvkey2.isin(universe)]
G = nx.from_pandas_edgelist(sub, 'gvkey1', 'gvkey2', edge_attr='score')
G.remove_edges_from(nx.selfloop_edges(G))
print(G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")

names = dict(zip(m.gvkey.astype(int), m.conm))
nx.set_node_attributes(G, names, 'name')

4099 seeds -> 4099 universe
3323 nodes, 248906 edges


In [69]:
comp[['gvkey','cik']].dropna(subset=['cik']).to_csv('../data/ciks_2023.csv', index=False)

In [79]:
# CIK code extraction for 2021
comp_21 = db.raw_sql("""
    SELECT gvkey, datadate, conm, cik, naicsh, sich, at, sale
    FROM comp.funda
    WHERE datadate BETWEEN '2021-01-01' AND '2021-12-31'
      AND indfmt='INDL' AND datafmt='STD'
      AND popsrc='D' AND consol='C'
""")
comp_21 = comp_21.drop_duplicates(subset=['cik','gvkey'])
comp_21.to_csv("../data/compustat_2021.csv", index=False)
comp_21[['gvkey','cik']].dropna(subset=['cik']).to_csv('../data/ciks_2021.csv', index=False)

In [80]:
comp_22 = db.raw_sql("""
    SELECT gvkey, datadate, conm, cik, naicsh, sich, at, sale
    FROM comp.funda
    WHERE datadate BETWEEN '2022-01-01' AND '2022-12-31'
      AND indfmt='INDL' AND datafmt='STD'
      AND popsrc='D' AND consol='C'
""")
comp_22 = comp_22.drop_duplicates(subset=['cik','gvkey'])
comp_22.to_csv("../data/compustat_2022.csv", index=False)
comp_22[['gvkey','cik']].dropna(subset=['cik']).to_csv('../data/ciks_2022.csv', index=False)

In [81]:
comp_24 = db.raw_sql("""
    SELECT gvkey, datadate, conm, cik, naicsh, sich, at, sale
    FROM comp.funda
    WHERE datadate BETWEEN '2024-01-01' AND '2024-12-31'
      AND indfmt='INDL' AND datafmt='STD'
      AND popsrc='D' AND consol='C'
""")
comp_24 = comp_24.drop_duplicates(subset=['cik','gvkey'])
comp_24.to_csv("../data/compustat_2024.csv", index=False)
comp_24[['gvkey','cik']].dropna(subset=['cik']).to_csv('../data/ciks_2024.csv', index=False)

In [82]:
comp_25 = db.raw_sql("""
    SELECT gvkey, datadate, conm, cik, naicsh, sich, at, sale
    FROM comp.funda
    WHERE datadate BETWEEN '2025-01-01' AND '2025-12-31'
      AND indfmt='INDL' AND datafmt='STD'
      AND popsrc='D' AND consol='C'
""")
comp_25 = comp_25.drop_duplicates(subset=['cik','gvkey'])
comp_25.to_csv("../data/compustat_2025.csv", index=False)
comp_25[['gvkey','cik']].dropna(subset=['cik']).to_csv('../data/ciks_2025.csv', index=False)

In [83]:
comp_26= db.raw_sql("""
    SELECT gvkey, datadate, conm, cik, naicsh, sich, at, sale
    FROM comp.funda
    WHERE datadate BETWEEN '2026-01-01' AND '2026-08-15'
      AND indfmt='INDL' AND datafmt='STD'
      AND popsrc='D' AND consol='C'
""")
comp_26 = comp_26.drop_duplicates(subset=['cik','gvkey'])
comp_26.to_csv("../data/compustat_2026.csv", index=False)
comp_26[['gvkey','cik']].dropna(subset=['cik']).to_csv('../data/ciks_2026.csv', index=False)

In [84]:
for y in [2021,2022,2023,2024,2025,2026]:
    c = db.raw_sql(f"""
        SELECT gvkey, datadate, conm, cik, naicsh, sich, at, sale
        FROM comp.funda
        WHERE datadate BETWEEN '{y}-01-01' AND '{y}-12-31'
          AND indfmt='INDL' AND datafmt='STD' AND popsrc='D' AND consol='C'
    """)
    c = c.sort_values('datadate').drop_duplicates('gvkey', keep='last')
    c[['gvkey','cik']].dropna(subset=['cik']).to_csv(f'ciks_{y}.csv', index=False)
    print(y, len(c))

2021 12286
2022 12574
2023 12701
2024 12922
2025 13429
2026 763


In [87]:
for y in range(2021, 2026):
    log = pd.read_csv(f"fetch_log_{y}.csv", header=None,
                      names=['gvkey','cik','adsh','rdate','status'])
    n_files = len(os.listdir(f"D:/dissertation/data/filings/{y}"))
    print(y, "files:", n_files, "| requested:", len(log))
    print("   ", log.status.value_counts().to_dict())

2021 files: 5208 | requested: 8231
    {'10-K': 5208, 'no_10k_for_year': 3017, 'submissions_failed': 6}
2022 files: 5143 | requested: 8167
    {'10-K': 5143, 'no_10k_for_year': 3021, 'submissions_failed': 3}
2023 files: 4937 | requested: 7963
    {'10-K': 4937, 'no_10k_for_year': 3023, 'submissions_failed': 3}
2024 files: 4762 | requested: 7591
    {'10-K': 4762, 'no_10k_for_year': 2826, 'submissions_failed': 3}
2025 files: 4507 | requested: 7056
    {'10-K': 4507, 'no_10k_for_year': 2546, 'submissions_failed': 3}


In [88]:
import glob
sizes = [os.path.getsize(f) for f in glob.glob("D:/dissertation/data/filings/2023/*.html")]
print(min(sizes), pd.Series(sizes).median(), max(sizes))

69091 2662411.0 31571450


In [89]:
log = pd.read_csv("fetch_log_2023.csv", header=None,
                  names=['gvkey','cik','adsh','rdate','status'])
miss = set(log[log.status=='no_10k_for_year'].gvkey)

c = pd.read_csv("../data/compustat_2023.csv")
print(c[c.gvkey.isin(miss)].sich.value_counts().head(10))
print(c[c.gvkey.isin(miss)].conm.head(15).tolist())

sich
6020.0    156
7370.0    125
2836.0    109
1040.0     77
2834.0     64
1000.0     52
7372.0     52
4412.0     43
8200.0     40
1311.0     37
Name: count, dtype: int64
['ASA GOLD AND PRECIOUS METALS', 'ACMAT CORP  -CL A', 'ADAMS DIVERSIFIED EQUITY FD', 'ASM INTERNATIONAL NV', 'AGNICO EAGLE MINES LTD', 'ALGOMA STEEL GROUP INC', 'AMERICAN BILTRITE INC', 'INVESCO BOND FUND', 'ATCO LTD', 'BRITISH AMER TOBACCO PLC', 'BARRICK MINING CORP', 'BCE INC', 'BLUE RIDGE REAL ESTATE CO', 'BROOKFIELD CORP', 'TELUS CORP']


In [90]:
have = {int(f.split('_')[0]) for f in os.listdir("D:/dissertation/data/filings/2023")}
tnic = set(df.gvkey1.unique())
print(len(tnic & have), "of", len(tnic))

4058 of 4099


In [96]:
UA = {"User-Agent": "dtomman1@binghamton.edu"}
miss_ciks = c[c.gvkey.isin(miss) & c.sich.isin([6020, 7370])].head(5)
for _, r in miss_ciks.iterrows():
    d = requests.get(f"https://data.sec.gov/submissions/CIK{str(int(r.cik)).zfill(10)}.json",
                     headers=UA).json()
    rec = d['filings']['recent']
    forms = pd.Series(rec['form'])
    print(r.conm, "|", forms.value_counts().head(4).to_dict())
    tenk = [(f, dt) for f, dt in zip(rec['form'], rec['reportDate']) if f.startswith('10-K')]
    print("   10-Ks:", tenk[:4])

SUMITOMO MITSUI FINANCIAL GR | {'6-K': 517, 'SUPPL': 42, '424B2': 40, '13F-HR': 39}
   10-Ks: []
BARCLAYS PLC | {'6-K': 233, 'SCHEDULE 13G/A': 186, 'SCHEDULE 13G': 156, 'SC 13G': 87}
   10-Ks: []
BANCO SANTANDER SA | {'6-K': 637, '424B5': 51, '13F-HR': 38, '425': 37}
   10-Ks: []
CADENCE BANK | {'13F-NT': 68, '13F-HR': 33, '144': 19, 'SC 13G/A': 16}
   10-Ks: []
NATIONAL AUSTRALIA BK | {'6-K': 220, 'ABS-15G': 9, '20-F': 5, 'UPLOAD': 4}
   10-Ks: []
